**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion II: Score Matching & SDEs

The rigorous sequel [Diffusion Models](./Diffusion_Models.ipynb) gestured at: what the network *actually* learns is the **score** $\nabla_x \log p(x)$, the discrete chain is an [SDE](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) in disguise, and generation is that SDE run backwards. Verified on a Gaussian mixture where the true score is available in closed form — the oracle most tutorials never check.

## 1. Pre-requisites

[Diffusion Models](./Diffusion_Models.ipynb) (the practical loop), [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) S4 (Brownian motion, Itô).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# ground-truth distribution: a 2-D Gaussian mixture — score known in CLOSED FORM
centers = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.2]])
sig2 = 0.35**2
def sample_data(n):
    idx = torch.randint(0, 3, (n,))
    return centers[idx] + 0.35*torch.randn(n, 2)
def true_score(x, t_var=0.0):
    """∇ log p_t for the mixture convolved with N(0, t_var) — exact."""
    v = sig2 + t_var
    d2 = ((x[:, None, :] - centers[None])**2).sum(-1)
    w = torch.softmax(-d2/(2*v), dim=1)
    mu_post = (w[:, :, None] * centers[None]).sum(1)
    return (mu_post - x) / v

---
### 🕐 Session 1 of 3 — *The Score Function* (~40 min)
**Goal:** learn ∇ log p by denoising; verify against the closed-form mixture score.
**Builds on:** [Diffusion Models](./Diffusion_Models.ipynb). &nbsp; **Feeds into:** Session 2 (Langevin & SDEs).

---

## 2. The Gradient of the Log-Density

💡 **Intuition.** The score $s(x) = \nabla_x \log p(x)$ is a *compass field*: at every point it points toward higher probability. You never need the (intractable) normalizing constant — gradients of $\log p$ kill it. And the miracle that makes it learnable: **denoising score matching** — training a network to predict the noise added to data is, up to scale, training it to output the score of the *noised* distribution ($s = -\varepsilon/\sigma$). Diffusion I's 'predict the noise' loss was secretly score estimation all along. Here we can *prove* it: our mixture's score has a closed form to compare against.

In [ ]:
# train a denoiser at ONE noise level; compare its implied score with the exact one
# ORACLE: implied score −net(x)/σ vs the closed-form mixture score at this noise level

# YOUR CODE HERE


---
### 🕐 Session 2 of 3 — *Langevin Dynamics & the Forward SDE* (~40 min)
**Goal:** climb the score with noise: Langevin sampling; the diffusion chain as an SDE.
**Builds on:** Session 1; [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb). &nbsp; **Feeds into:** Session 3 (the reverse SDE & probability flow).

---

## 3. Sampling = Noisy Gradient Ascent

💡 **Intuition.** Given a score, **Langevin dynamics** samples: $x \mathrel{+}= \frac{\eta}{2} s(x) + \sqrt{\eta}\, \xi$ — climb the compass field, but inject just enough noise that you *explore* the distribution instead of collapsing to its modes ([SGD's noise](../Intro_Math/Optimization/Optimization.ipynb), now load-bearing). The catch that motivated diffusion: with far-apart modes, plain Langevin mixes badly — which is why diffusion runs a *family* of scores across noise levels: high noise merges the modes for easy travel, low noise sharpens the details. And in the continuum, Diffusion I's chain **is** the SDE $dx = -\tfrac12\beta x\, dt + \sqrt{\beta}\, dB$ — an Ornstein–Uhlenbeck process whose marginals we can check exactly.

In [ ]:
# ORACLE: the forward SDE's variance must follow the OU closed form

# YOUR CODE HERE


---
### 🕐 Session 3 of 3 — *The Reverse SDE & Probability Flow* (~40 min)
**Goal:** run time backwards with the score; sample the mixture and audit mode weights.
**Builds on:** Session 2.

---

## 4. Anderson's Time Machine

💡 **Intuition.** The stunning theorem (Anderson, 1982): the forward SDE has an exact **reverse**: $dx = [-\tfrac12\beta x - \beta\, s_t(x)]\, dt + \sqrt{\beta}\, d\bar{B}$ — identical dynamics plus a score-guided drift. Everything unknown about 'undoing noise' is packed into $s_t$, the exact object Session 1 taught us to learn. Drop the noise term and halve the score drift and you get the **probability-flow ODE** — deterministic, same marginals, the bridge to flow matching and fast samplers (DDIM is its discretization).

In [ ]:
# train a time-conditional score net, then sample by reverse SDE — audit the result
# reverse SDE from pure noise
# ORACLE: mode weights should be ≈ 1/3 each; mode means ≈ the centers

# YOUR CODE HERE


## 5. Conclusion

'Predict the noise' is score estimation (verified against a closed-form score, cosine ≈ 1); the chain is an OU SDE (variance audited against Itô); and Anderson's reverse SDE turns the learned compass into a sampler whose mode weights and centers match the truth. Diffusion I's recipe now has its complete mathematical spine.

---
## Where next

- [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) — the Itô calculus underneath.
- [Optimal Transport](./Optimal_Transport.ipynb) — probability flow as a transport map; flow matching lives here.